In [5]:
import pandas as pd
import numpy as np

In [6]:
def run_boosting_ensemble_ppt_style(model_files, k=8):
    """
    依照 PPT 範例實作：迭代更新題目權重 w_i 與模型權重 alpha_j
    """
    models_data = {}
    for name, file in model_files.items():
        models_data[name] = pd.read_excel(file)
    
    num_instances = len(next(iter(models_data.values())))
    actual_labels = next(iter(models_data.values()))['Actual']
    
    # 1. 初始設定：題目初始權重每題平等 [cite: 619]
    instance_weights = np.ones(num_instances) / num_instances
    
    # 儲存各模型最終的累積權重 (用於最終決策) [cite: 663-666]
    final_cumulative_alphas = {name: 0.0 for name in model_files.keys()}

    print(f"--- 執行 Boosting 迭代 (設定輪數 k={k}) ---")
    print(f"初始題目權重分布: {instance_weights}")

    for round_idx in range(1, k + 1):
        print(f"\n【第 {round_idx} 輪迭代】")
        
        for name, df in models_data.items():
            # 取得該模型預測錯誤的遮罩 (1為錯, 0為對) [cite: 621-622]
            errors = (df['Correct'] == False).astype(int)
            
            # 2. 計算加權誤差 Ej [cite: 625-626, 635-636]
            # Ej = (做錯題目的權重和) / (所有題目的權重和)
            weighted_error = np.dot(instance_weights, errors) / np.sum(instance_weights)
            weighted_error = max(min(weighted_error, 0.999), 0.001)
            
            # 3. 計算本輪投票權重 alpha_j [cite: 627-628, 637-638]
            alpha = 0.5 * np.log((1 - weighted_error) / weighted_error) * num_instances * 0.0001
            
            # 4. 更新題目權重: 做錯的題目權重增加 alpha [cite: 629-630, 639-640, 673]
            # 注意：範例中是直接 w_i <- w_i + alpha
            instance_weights = instance_weights + alpha * errors
            
            # 紀錄本輪模型權重
            final_cumulative_alphas[name] = alpha # += alpha
            
            print(f"  > 評估 {name}:")
            print(f"    - 加權誤差 Ej: {weighted_error:.4f}")
            print(f"    - 得到 Alpha: {alpha:.4f} (累積: {final_cumulative_alphas[name]:.4f})")

        # 每輪結束後計算當前正確率 
        current_correct = 0
        for idx in range(num_instances):
            votes = {}
            for name in final_cumulative_alphas:
                pred = models_data[name].loc[idx, 'Predicted']
                votes[pred] = votes.get(pred, 0) + final_cumulative_alphas[name]
            
            if max(votes, key=votes.get) == actual_labels[idx]:
                current_correct += 1
        
        acc = (current_correct / num_instances) * 100
        print(f"  > 本輪結束題目權重分布 (前三題): {instance_weights[:3]}")
        print(f"  > 目前集成正確率: {current_correct}/{num_instances} = {acc:.2f}%")

    return final_cumulative_alphas, models_data

In [7]:
# --- 主程式執行 ---
files = {
    "M1": "result_qwen3_8b.xlsx",
    "M2": "result_huihui_ai_internlm3-abliterated.xlsx",
    "M3": "result_gemma3_12b.xlsx"
}

In [8]:
# 依照範例設定 k=2
final_weights, data = run_boosting_ensemble_ppt_style(files, k=20)

--- 執行 Boosting 迭代 (設定輪數 k=20) ---
初始題目權重分布: [0.0005291 0.0005291 0.0005291 ... 0.0005291 0.0005291 0.0005291]

【第 1 輪迭代】
  > 評估 M1:
    - 加權誤差 Ej: 0.0725
    - 得到 Alpha: 0.2409 (累積: 0.2409)
  > 評估 M2:
    - 加權誤差 Ej: 0.5052
    - 得到 Alpha: -0.0020 (累積: -0.0020)
  > 評估 M3:
    - 加權誤差 Ej: 0.5050
    - 得到 Alpha: -0.0019 (累積: -0.0019)
  > 本輪結束題目權重分布 (前三題): 0    0.000529
1    0.000529
2    0.000529
Name: Correct, dtype: float64
  > 目前集成正確率: 1753/1890 = 92.75%

【第 2 輪迭代】
  > 評估 M1:
    - 加權誤差 Ej: 0.9803
    - 得到 Alpha: -0.3694 (累積: -0.3694)
  > 評估 M2:
    - 加權誤差 Ej: 0.5518
    - 得到 Alpha: -0.0197 (累積: -0.0197)
  > 評估 M3:
    - 加權誤差 Ej: 0.5492
    - 得到 Alpha: -0.0187 (累積: -0.0187)
  > 本輪結束題目權重分布 (前三題): 0    0.000529
1    0.000529
2    0.000529
Name: Correct, dtype: float64
  > 目前集成正確率: 1719/1890 = 90.95%

【第 3 輪迭代】
  > 評估 M1:
    - 加權誤差 Ej: 0.9103
    - 得到 Alpha: -0.2190 (累積: -0.2190)
  > 評估 M2:
    - 加權誤差 Ej: 0.5542
    - 得到 Alpha: -0.0206 (累積: -0.0206)
  > 評估 M3:
    - 加權誤差 Ej: 0.5533
    -